#  Arm Assembly Primer : Calling a function
# 1. Calling and returning
* Here `add2` calls `add1` twice. `add1` is only _declared_ here (its definition is in another file), so the compiler cannot see what it does and must really call it


In [ ]:
%%writefile call.c
int add1(int x);   /* defined elsewhere */

int add2(int x) {
  return add1(add1(x));
}

int add1_again(int x) {
  return add1(x);
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S call.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' call.s

* You will see something like this

```
add2:
	stp	x29, x30, [sp, #-16]!
	mov	x29, sp
	bl	add1
	ldp	x29, x30, [sp], #16
	b	add1
add1_again:
	b	add1
```

* <font color="blue">`bl add1`</font> (_branch with link_) calls `add1`
  * it jumps to the label `add1`, and
  * it puts the address of the next instruction (the _return address_) into register `x30`, so that `add1` can come back
* <font color="blue">`ret`</font> (at the end of `add1`, which we saw in the previous notebook) jumps to the address in `x30`, i.e., back to the instruction after `bl add1`
* `x30` is also called the _link register_ (`lr`)

# 2. Why `stp` and `ldp`?
* `add2` itself was called by someone, with _its_ return address in `x30`
* `bl add1` overwrites `x30`, so `add2` must save the old value before calling and restore it afterwards; otherwise it could not return to its caller
* `stp x29, x30, [sp, #-16]!` saves (stores) `x29` and `x30` into memory, in the area called the _stack_
  * `sp` (_stack pointer_) is a register holding the address of the top of the stack
  * `[sp, #-16]!` means "decrease `sp` by 16 (bytes), and store the pair of registers at the address `sp`"
  * `x29` (the _frame pointer_) is saved and set (`mov x29, sp`) by convention, so that debuggers can follow the chain of calls
* `ldp x29, x30, [sp], #16` loads them back and increases `sp` by 16 (bytes), undoing the `stp`
* `add1_again` does not need them; see below

# 3. Tail calls
* The second call to `add1` in `add2` is not `bl add1` followed by `ret`, but just <font color="blue">`b add1`</font>, a plain jump (_branch_) that does not touch `x30`
* This works because the call is the last thing `add2` does: when `add1` executes `ret`, it jumps to the address in `x30`, which by then is the return address of `add2` itself (restored by `ldp`). So `add1` returns directly to `add2`'s caller
* This optimization is called a _tail call_. `add1_again` consists of only `b add1` for the same reason

# 4. Inlining
* If the compiler can see the body of the function, it often does not call it at all; it copies the body into the caller instead (_inlining_)


In [ ]:
%%writefile call_inline.c
int add1(int x) {
  return x + 1;
}

int add2(int x) {
  return add1(add1(x));
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S call_inline.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' call_inline.s

* `add2` is now just `add w0, w0, #2` and `ret`; `add1` is still there (another file might call it), but `add2` does not call it
* This is why we often declare a function without defining it (as in `call.c`) in the examples that follow: to make the compiler really call it

# 5. Summary
* `bl f` calls `f`: jump to `f` and put the return address in `x30`
* `ret` returns: jump to the address in `x30`
* a function that calls another function saves `x30` on the stack first (`stp`) and restores it (`ldp`) before returning
* `b f` is a plain jump; it is used for a call at the very end of a function (tail call)
* functions whose bodies are visible are often inlined, and you will not see calls to them